In [6]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [10]:
client.search_experiments(max_results=1)[0]

<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/3', creation_time=1747703946661, experiment_id='3', last_update_time=1747703946661, lifecycle_stage='active', name='my-experiment', tags={}>

In [95]:
client.create_experiment(name="my-experiment")

MlflowException: Experiment(name=my-experiment) already exists. Error: (raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.IntegrityError) UNIQUE constraint failed: experiments.name
[SQL: INSERT INTO experiments (name, artifact_location, lifecycle_stage, creation_time, last_update_time) VALUES (?, ?, ?, ?, ?)]
[parameters: ('my-experiment', None, 'active', 1747758349023, 1747758349023)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [96]:
client.search_experiments()

[<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/3', creation_time=1747703946661, experiment_id='3', last_update_time=1747703946661, lifecycle_stage='active', name='my-experiment', tags={}>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/2', creation_time=1747582843607, experiment_id='2', last_update_time=1747582843607, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/0', creation_time=1747312776094, experiment_id='0', last_update_time=1747312776094, lifecycle_stage='active', name='Default', tags={}>]

In [97]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='2',
    filter_string="metrics.rmse < 6.5",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [98]:
for run in runs:
    print(f"run id:{run.info.run_id}, rmse:{run.data.metrics['rmse']:.4f}")


run id:4fb4988e930640e9b6db5498e9613823, rmse:6.3057
run id:68f5e9e442814643a14f971d0c506484, rmse:6.3090
run id:b3afea64162c46769071d09f3a613616, rmse:6.3304
run id:fd87b6a9e19c4ad995174b4253b4ae78, rmse:6.3318
run id:bea68c63e3694c2da9c5e05de5fc3e1c, rmse:6.3323


In [99]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [100]:
run_id = "bea68c63e3694c2da9c5e05de5fc3e1c"
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
Created version '8' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1747758359986, current_stage='None', description=None, last_updated_timestamp=1747758359986, name='nyc-taxi-regressor', run_id='bea68c63e3694c2da9c5e05de5fc3e1c', run_link=None, source='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/2/bea68c63e3694c2da9c5e05de5fc3e1c/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=8>

In [101]:
client.search_registered_models()

[<RegisteredModel: aliases={'Production': 7}, creation_timestamp=1747690998542, description='The NYC taxi Predictor for Trip Duration', last_updated_timestamp=1747758359986, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1747758359986, current_stage='None', description=None, last_updated_timestamp=1747758359986, name='nyc-taxi-regressor', run_id='bea68c63e3694c2da9c5e05de5fc3e1c', run_link=None, source='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/2/bea68c63e3694c2da9c5e05de5fc3e1c/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=8>], name='nyc-taxi-regressor', tags={}>]

In [102]:
model_name = 'nyc-taxi-regressor'
latest_versions = client.get_latest_versions(name=model_name)
for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 8, stage: None


/tmp/ipykernel_51521/127348374.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [103]:
model_version = 7
alias = "Production"
client.set_registered_model_alias(
    name=model_name,
    alias=alias,
    version=model_version)

In [104]:
from datetime import datetime
date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {alias} on {date}"
)

<ModelVersion: aliases=['Production'], creation_timestamp=1747755299865, current_stage='None', description='The model version 7 was transitioned to Production on 2025-05-20', last_updated_timestamp=1747758360861, name='nyc-taxi-regressor', run_id='bea68c63e3694c2da9c5e05de5fc3e1c', run_link=None, source='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/2/bea68c63e3694c2da9c5e05de5fc3e1c/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=7>

In [105]:
from sklearn.metrics import root_mean_squared_error
import mlflow.xgboost
import pandas as pd

def read_dataframe(filename):
    df = pd.read_parquet(filename)
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[(df.duration >=1) & (df.duration <= 60)]
    categorical = ['PULocationID','DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df

def preprocess(df,dv):
    df['PU_DO'] = df['PULocationID'] + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    test_dict = df[categorical+numerical].to_dict(orient='records')
    return dv.fit_transform(test_dict)

def test_model(alias, X_test, y_test, name="nyc-taxi-regressor"):
    model = mlflow.xgboost.load_model(f"models:/{name}@{alias}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test,y_pred)}






In [106]:
df = read_dataframe("data/green_tripdata_2021-03.parquet")

In [107]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path = '.')

'/workspaces/mlops-zoomcamp/02-experiment-tracking/preprocessor'

In [108]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [109]:
import xgboost as xgb
X_test = preprocess(df, dv)
X_test = xgb.DMatrix(X_test)

In [110]:
target = "duration"
y_test = df[target].values

In [112]:
X_test

In [111]:
%time test_model(name = model_name, alias='Production', X_test=X_test,y_test=y_test)

XGBoostError: [16:26:05] /workspace/src/learner.cc:1462: Check failed: learner_model_param_.num_feature >= p_fmat->Info().num_col_ (13063 vs. 13617) : Number of columns does not match number of features in booster.
Stack trace:
  [bt] (0) /home/codespace/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/xgboost/lib/libxgboost.so(+0x25c1ac) [0x77728c86d1ac]
  [bt] (1) /home/codespace/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/xgboost/lib/libxgboost.so(+0x5e64c9) [0x77728cbf74c9]
  [bt] (2) /home/codespace/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/xgboost/lib/libxgboost.so(+0x5f8441) [0x77728cc09441]
  [bt] (3) /home/codespace/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/xgboost/lib/libxgboost.so(XGBoosterPredictFromDMatrix+0x2a8) [0x77728c77b0d8]
  [bt] (4) /home/codespace/anaconda3/envs/exp-tracking-env/lib/python3.9/lib-dynload/../../libffi.so.8(+0xa052) [0x77731c98d052]
  [bt] (5) /home/codespace/anaconda3/envs/exp-tracking-